In [ ]:
!pip install datasets==3.6.0 conllu transformers evaluate seqeval accelerate -q

In [ ]:
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

import evaluate
import torch

In [ ]:
print("GPU:", torch.cuda.is_available())

GPU: True


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "universal_dependencies",
    "en_ewt",
    trust_remote_code=True
)

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/12543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2002
    })
    test: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2077
    })
})

In [ ]:
dataset["train"][0]

{'idx': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001',
 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.',
 'tokens': ['Al',
  '-',
  'Zaman',
  ':',
  'American',
  'forces',
  'killed',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
  'the',
  'Syrian',
  'border',
  '.'],
 'lemmas': ['Al',
  '-',
  'Zaman',
  ':',
  'american',
  'force',
  'kill',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
  'the',
  'syrian',
  'border',
  '.'],
 'upos': [10,
  1,
  10,
  1,
  6,
  0,
  16,
  10,
  10,
  10,
  1,
  10,
  1,
  8,
  0,
  2,
  8,
  0,
  2,
  8,
  0,
  2,
  10,
  1,
  2,
  8,
  6,
  0,
  1],
 'xpos': ['NNP',
  'HYPH',
  'NNP

In [ ]:
label_names = dataset["train"].features["upos"].feature.names

print("Number of Labels:", len(label_names))
print("Labels:", label_names)

Number of Labels: 18
Labels: ['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']


# Task 1: Dataset Selection

**Dataset Name:** Universal Dependencies (English EWT)

**Task:** Part-of-Speech (POS) Tagging

**Label Categories:**
- NOUN
- VERB
- ADJ
- ADV
- PRON
- DET
- ADP
- PROPN
- etc.

In [ ]:
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []

    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_dataset

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2002
    })
    test: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2077
    })
})

In [ ]:
tokenized_dataset["train"][0]

{'idx': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001',
 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.',
 'tokens': ['Al',
  '-',
  'Zaman',
  ':',
  'American',
  'forces',
  'killed',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
  'the',
  'Syrian',
  'border',
  '.'],
 'lemmas': ['Al',
  '-',
  'Zaman',
  ':',
  'american',
  'force',
  'kill',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
  'the',
  'syrian',
  'border',
  '.'],
 'upos': [10,
  1,
  10,
  1,
  6,
  0,
  16,
  10,
  10,
  10,
  1,
  10,
  1,
  8,
  0,
  2,
  8,
  0,
  2,
  8,
  0,
  2,
  10,
  1,
  2,
  8,
  6,
  0,
  1],
 'xpos': ['NNP',
  'HYPH',
  'NNP

In [ ]:
num_labels = len(label_names)

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(id2label)

{0: 'NOUN', 1: 'PUNCT', 2: 'ADP', 3: 'NUM', 4: 'SYM', 5: 'SCONJ', 6: 'ADJ', 7: 'PART', 8: 'DET', 9: 'CCONJ', 10: 'PROPN', 11: 'PRON', 12: 'X', 13: '_', 14: 'ADV', 15: 'INTJ', 16: 'VERB', 17: 'AUX'}


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
metric = evaluate.load("seqeval")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for pred, lab in zip(prediction, label):
            if lab != -100:
                current_preds.append(label_names[pred])
                current_labels.append(label_names[lab])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./pos_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.342032,0.152703,0.951051,0.955034,0.953038,0.958906
2,0.093574,0.139675,0.957169,0.960448,0.958806,0.964047
3,0.066414,0.139646,0.957987,0.960702,0.959343,0.964558


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2352, training_loss=0.16734000614711217, metrics={'train_runtime': 474.8394, 'train_samples_per_second': 79.246, 'train_steps_per_second': 4.953, 'total_flos': 1229441949457920.0, 'train_loss': 0.16734000614711217, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

{'eval_loss': 0.1396462321281433, 'eval_precision': 0.9579870924199604, 'eval_recall': 0.9607021996615905, 'eval_f1': 0.9593427249878556, 'eval_accuracy': 0.9645576575869378, 'eval_runtime': 8.4702, 'eval_samples_per_second': 236.357, 'eval_steps_per_second': 14.876, 'epoch': 3.0}


In [ ]:
def predict_pos(sentence):
    words = sentence.split()

    inputs = tokenizer(
        words,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True,
        padding=True
    )

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    predicted_tags = []

    word_ids = inputs.word_ids()

    previous_word_idx = None

    for pred, word_idx in zip(predictions[0], word_ids):
        if word_idx is not None and word_idx != previous_word_idx:
            predicted_tags.append(id2label[pred.item()])
        previous_word_idx = word_idx

    return list(zip(words, predicted_tags))

In [ ]:
def predict_pos(sentence):
    words = sentence.split()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(
        words,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True,
        padding=True
    )

    # Move all input tensors to same device as model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    predicted_tags = []

    # word_ids must be taken BEFORE moving to device, so regenerate tokenizer output for alignment
    temp_inputs = tokenizer(
        words,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True,
        padding=True
    )

    word_ids = temp_inputs.word_ids()

    previous_word_idx = None

    for pred, word_idx in zip(predictions[0], word_ids):
        if word_idx is not None and word_idx != previous_word_idx:
            predicted_tags.append(id2label[pred.item()])
        previous_word_idx = word_idx

    return list(zip(words, predicted_tags))

In [ ]:
sentence = "John works at Google in California"

result = predict_pos(sentence)

for word, tag in result:
    print(f"{word}: {tag}")

John: PROPN
works: VERB
at: ADP
Google: PROPN
in: ADP
California: PROPN


# Task 7: POS Tagging vs Chunking

| Feature | POS Tagging | Chunking |
|---|---|---|
| Level | Word | Phrase |
| Difficulty | Easy | Medium |
| Output | NOUN, VERB | NP, VP |
| Purpose | Grammar role | Phrase grouping |

Example:

Sentence: John works at Google

POS:
John → PROPN  
works → VERB  

Chunking:
[John] NP  
[works at Google] VP

# Task 8: Report / Blog

## Differences
POS tagging assigns grammatical labels to each word.

Chunking groups words into meaningful phrases.

## Challenges Faced
- Subword token alignment
- Special token handling
- Proper masking using -100
- GPU memory optimization

## Observations
DistilBERT performs highly well on sequence labeling tasks.

The model achieved strong precision, recall, and F1 score.

In [ ]:
trainer.save_model("final_pos_model")
tokenizer.save_pretrained("final_pos_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_pos_model/tokenizer_config.json', 'final_pos_model/tokenizer.json')

In [ ]:
from google.colab import files
files.download("final_pos_model/config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>